In [0]:
#%run ./python_libraries ----- A décommmenter pour lancer les notebooks séparements

In [0]:
#%run ../delta_function ----- A décommmenter pour lancer les notebooks séparements

In [0]:
#%run ./env

In [0]:
# Définir la source en fonction de l'environnement
if current_environment =='preprd':
    source_catalog = f"""ext_mal_psql_maite_vision_board_test.public"""    
else:
    source_catalog = f"""ext_mal_psql_maite_vision_board_{current_environment}.public"""

In [0]:
catalog_measurements_prd = f"""mal_maite_common_{current_environment}.gold"""
catalog_tps_process = f"""mal_maite_bi_{current_environment}.process_time_analyses"""

In [0]:
#measurements_preprd = spark.table(f"{catalog_measurements_preprd}.measurements") 
#measurement = spark.table(f"{catalog_measurements_prd}.measurements_pivot") 
localisation = spark.table(f"{catalog_measurements_prd}.localizations_pivot")
species = spark.table(f"{source_catalog}.goods_species")
varieties = spark.table(f"{source_catalog}.goods_varieties")
batches = spark.table(f"{source_catalog}.batches")
plants_production_lines = spark.table(f"{source_catalog}.plants_production_lines")
requirement_specifications = spark.table(f"{source_catalog}.requirement_specifications")
manual_entries = spark.table(f"{source_catalog}.manual_entries")
parameters_variables = spark.table(f"{source_catalog}.parameters_variables")

parameters_cycles = spark.table(f"{source_catalog}.parameters_batch_cycles")

#processes_phases = spark.table(f"{source_catalog}.processes_phases")
batch_note = spark.table(f"{source_catalog}.batches_notes")
#batch_note_translation = spark.table(f"{source_catalog}.parameters_batch_note_categories_translations")
production_type = spark.table(f"{source_catalog}.parameters_production_types")
profiles_users = spark.table(f"{source_catalog}.profiles_users")

# tables d'ascendance (ajout prd_line)
ascendance_ro1 = spark.table(f"mal_maite_rouen1_{current_environment}.monitoring.historian_ancestry")
ascendance_ng1 = spark.table(f"mal_maite_nogent1_{current_environment}.monitoring.historian_ancestry")
ascendance_ng2 = spark.table(f"mal_maite_nogent2_{current_environment}.monitoring.historian_ancestry")
ascendance_pr1 = spark.table(f"mal_maite_prouvy1_{current_environment}.monitoring.historian_ancestry")
ascendance_st1 = spark.table(f"mal_maite_strasbourg1_{current_environment}.monitoring.historian_ancestry")
ascendance_st2 = spark.table(f"mal_maite_strasbourg2_{current_environment}.monitoring.historian_ancestry")
ascendance_po1 = spark.table(f"mal_maite_polisy1_{current_environment}.monitoring.historian_ancestry")
ascendance_bu1 = spark.table(f"mal_maite_buzau1_{current_environment}.monitoring.historian_ancestry")
ascendance_bo1 = spark.table(f"mal_maite_bolelemi1_{current_environment}.monitoring.historian_ancestry")

#DM-5912
# Tables sources des mesures pré-calculées par site - Optimisé : factorisation en boucle
from pyspark.sql import functions as F

sites = ["rouen1", "nogent1", "nogent2", "prouvy1", "strasbourg1", "strasbourg2", "polisy1", "buzau1", "bolelemi1"]
mesures_filter = (F.col("measure.status") == "OK") & (F.col("measure.value").isNotNull())

mesures_calculees = {
    site: spark.table(f"mal_maite_{site}_{current_environment}.gold.measurements")
          .filter(mesures_filter)
    for site in sites
}

# Aliases pour compatibilité avec transform_data
mesures_calculees_ro1 = mesures_calculees["rouen1"]
mesures_calculees_ng1 = mesures_calculees["nogent1"]
mesures_calculees_ng2 = mesures_calculees["nogent2"]
mesures_calculees_pr1 = mesures_calculees["prouvy1"]
mesures_calculees_st1 = mesures_calculees["strasbourg1"]
mesures_calculees_st2 = mesures_calculees["strasbourg2"]
mesures_calculees_po1 = mesures_calculees["polisy1"]
mesures_calculees_bu1 = mesures_calculees["buzau1"]
mesures_calculees_bo1 = mesures_calculees["bolelemi1"]

#DM-5912
# Table metadata 
metadata_mesures = spark.table(f"{catalog_measurements_prd}.trs_metadata")

####Amélioration 
#fact_unpivoted = spark.table(f"""{catalog_tps_process}.fact_process_time_mesures_unpivoted""") 
dim_batches_specifications = spark.table(f"""{catalog_tps_process}.dim_batches_specifications""")

## Tables de traduction (phase 2 — traduction des données)

Mêmes catalogues que les autres tables métier : elles vivent dans
`source_catalog`, la base PostgreSQL alimentée par l'équipe front.

Elles ne sont **pas** filtrées sur `deleted` ici : le filtrage est fait dans
`dim_translations`, après le dédoublonnage, pour rester cohérent avec le reste
du traitement.

In [0]:
# Référentiel des langues : porte le filtre RLS du modèle
parameters_languages = spark.table(f"{source_catalog}.parameters_languages")

# Tables de traduction du contenu (périmètre 🟡 du Document PO)
goods_species_translations = spark.table(f"{source_catalog}.goods_species_translations")
goods_varieties_translations = spark.table(f"{source_catalog}.goods_varieties_translations")
parameters_production_type_translations = spark.table(f"{source_catalog}.parameters_production_type_translations")
parameters_variables_translations = spark.table(f"{source_catalog}.parameters_variables_translations")
parameters_production_line_variables_translations = spark.table(f"{source_catalog}.parameters_production_line_variables_translations")
parameters_localizations_translations = spark.table(f"{source_catalog}.parameters_localizations_translations")
parameters_localization_groups_translations = spark.table(f"{source_catalog}.parameters_localization_groups_translations")
parameters_batch_note_categories_translations = spark.table(f"{source_catalog}.parameters_batch_note_categories_translations")